
# Analyzing Startup Fundraising Deals from Crunchbase

In this project, we will be analysing startup fundraising data from Crunchbase to explore investments, funding types, and amounts raised. We will use the `pandas` and `sqlite3` libraries to practice handling large datasets efficiently under memory constraints, simulating real-world data analysis scenarios.

In [1]:
import pandas as pd
import sqlite3
investments=pd.read_csv("crunchbase-investments.csv", nrows=5, encoding="iso-8859-1")
investments

,company_permalink,company_name,company_category_code,company_country_code,company_state_code,company_region,company_city,investor_permalink,investor_name,investor_category_code,investor_country_code,investor_state_code,investor_region,investor_city,funding_round_type,funded_at,funded_month,funded_quarter,funded_year,raised_amount_usd
0,/company/advercar,AdverCar,advertising,USA,CA,SF Bay,San Francisco,/company/1-800-flowers-com,1-800-FLOWERS.COM,NaN,USA,NY,New York,New York,series-a,2012-10-30,2012-10,2012-Q4,2012,2000000
1,/company/launchgram,LaunchGram,news,USA,CA,SF Bay,Mountain View,/company/10xelerator,10Xelerator,finance,USA,OH,Columbus,Columbus,other,2012-01-23,2012-01,2012-Q1,2012,20000
2,/company/utap,uTaP,messaging,USA,NaN,United States - Other,NaN,/company/10xelerator,10Xelerator,finance,USA,OH,Columbus,Columbus,other,2012-01-01,2012-01,2012-Q1,2012,20000
3,/company/zoopshop,ZoopShop,software,USA,OH,Columbus,columbus,/company/10xelerator,10Xelerator,finance,USA,OH,Columbus,Columbus,angel,2012-02-15,2012-02,2012-Q1,2012,20000
4,/company/efuneral,eFuneral,web,USA,OH,Cleveland,Cleveland,/company/10xelerator,10Xelerator,finance,USA,OH,Columbus,Columbus,other,2011-09-08,2011-09,2011-Q3,2011,20000


In [2]:
chunk_iter=pd.read_csv("crunchbase-investments.csv", chunksize=5000, encoding="iso-8859-1")

no_chunks=0
no_rows=0
missing_values=[]
column_memory={}
total_memory=0

for chunk in chunk_iter:

    no_chunks+=1
    no_rows+=len(chunk)
    total_memory+=chunk.memory_usage(deep=True).sum()

    for col in chunk:
        if col in column_memory:
            column_memory[col]+=chunk[col].memory_usage(deep=True)
        else:
            column_memory[col]=chunk[col].memory_usage(deep=True)
    missing_values.append(chunk.isnull().sum())
combined_mv=pd.concat(missing_values)
unique_combined_mv=combined_mv.groupby(combined_mv.index).sum().sort_values()

print(f"No of chunks: {no_chunks}")
print(f"Total no of rows=: {no_rows}")
print(f"Total memory footprint of all chunks: {total_memory / (1024 ** 2):.2f} MB")
print(f"\nMemory usage per column:")
for col,mem in column_memory.items():
    print((col),":",round(mem/(1024**2),2),"MB")
print(f"\nMissing values per column:")
print(f"{unique_combined_mv}")

No of chunks: 11
Total no of rows=: 52870
Total memory footprint of all chunks: 50.44 MB

Memory usage per column:
company_permalink : 3.47 MB
company_name : 3.02 MB
company_category_code : 2.87 MB
company_country_code : 2.62 MB
company_state_code : 2.56 MB
company_region : 2.85 MB
company_city : 2.95 MB
investor_permalink : 4.35 MB
investor_name : 3.33 MB
investor_category_code : 0.58 MB
investor_country_code : 2.21 MB
investor_state_code : 2.09 MB
investor_region : 2.84 MB
investor_city : 2.44 MB
funding_round_type : 2.85 MB
funded_at : 2.98 MB
funded_month : 2.82 MB
funded_quarter : 2.82 MB
funded_year : 0.4 MB
raised_amount_usd : 0.4 MB

Missing values per column:
company_name                  1
company_country_code          1
company_region                1
company_permalink             1
investor_name                 2
investor_permalink            2
investor_region               2
funded_at                     3
funding_round_type            3
funded_quarter                3
fun

In [3]:
drop_cols=["company_permalink", "investor_permalink", "investor_category_code", "funded_month", "funded_year", "funded_quarter"]
useful_cols=chunk.columns.drop(drop_cols).to_list()

In [4]:
chunk_iter=pd.read_csv("crunchbase-investments.csv", chunksize=5000, encoding="iso-8859-1", usecols=useful_cols)
col_dtypes={}

inconsistent_cols=set()

for chunk in chunk_iter:
    for col in chunk.columns:
        if col not in col_dtypes:
            col_dtypes[col]=[str(chunk[col].dtype)]
        else:
            if str(chunk[col].dtype) not in col_dtypes[col]:
                col_dtypes[col].append(str(chunk[col].dtype))
                inconsistent_cols.add(col)

print("\nEach column's data types:")
for col,dtype in col_dtypes.items():
    print(col,":",dtype)

print("\n",inconsistent_cols)


Each column's data types:
company_name : ['str']
company_category_code : ['str']
company_country_code : ['str']
company_state_code : ['str']
company_region : ['str']
company_city : ['str']
investor_name : ['str']
investor_country_code : ['str', 'float64']
investor_state_code : ['str', 'float64']
investor_region : ['str']
investor_city : ['str', 'float64']
funding_round_type : ['str']
funded_at : ['str']
raised_amount_usd : ['float64']

 {'investor_city', 'investor_country_code', 'investor_state_code'}


In [5]:
column_types={'investor_country_code' : 'object', 'investor_state_code' : 'object',
              'investor_city' : 'object'}

chunk_iter=pd.read_csv("crunchbase-investments.csv", chunksize=5000,
                       encoding="iso-8859-1", usecols=useful_cols,
                      dtype=column_types)

unique_value={}     #The unique value counts from all the object columns

total_row=52870

for chunk in chunk_iter:
    for col in chunk.select_dtypes(include=['object','str']).columns.to_list():
        unique_values=chunk[col].value_counts()
        if col not in unique_value:
            unique_value[col]=[unique_values]
        else:
            unique_value[col].append(unique_values)

unique_value_count={}

for col,values in unique_value.items():
    combined=pd.concat(values)
    grouped=combined.groupby(combined.index).sum()
    unique_value_count[col]=len(grouped)

unique_value_count

{'company_name': 11573,
 'company_category_code': 43,
 'company_country_code': 2,
 'company_state_code': 50,
 'company_region': 546,
 'company_city': 1229,
 'investor_name': 10465,
 'investor_country_code': 72,
 'investor_state_code': 50,
 'investor_region': 585,
 'investor_city': 990,
 'funding_round_type': 9,
 'funded_at': 2808}

In [6]:
category_cols={'company_category_code': 43,
 'company_country_code': 2,
 'company_state_code': 50,
 'company_region': 546,
 'company_city': 1229,
 'investor_country_code': 72,
 'investor_state_code': 50,
 'investor_region': 585,
 'investor_city': 990,
 'funding_round_type': 9}

chunk_iter=pd.read_csv("crunchbase-investments.csv", chunksize=5000,
                       encoding="iso-8859-1", usecols=useful_cols,
                      dtype=column_types, parse_dates=["funded_at"])

overall_memory=0

for chunk in chunk_iter:
    for col in category_cols.keys():
        if col in chunk.columns:
            chunk[col]=chunk[col].astype('category')
    memory_usage=chunk.memory_usage(deep=True).sum()/1024**2
    overall_memory+=memory_usage


print(overall_memory)

chunk.info()

8.515127182006836
<class 'pandas.DataFrame'>
RangeIndex: 2870 entries, 50000 to 52869
Data columns (total 14 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   company_name           2870 non-null   str           
 1   company_category_code  2860 non-null   category      
 2   company_country_code   2870 non-null   category      
 3   company_state_code     2841 non-null   category      
 4   company_region         2870 non-null   category      
 5   company_city           2825 non-null   category      
 6   investor_name          2870 non-null   str           
 7   investor_country_code  0 non-null      category      
 8   investor_state_code    0 non-null      category      
 9   investor_region        2870 non-null   category      
 10  investor_city          0 non-null      category      
 11  funding_round_type     2870 non-null   category      
 12  funded_at              2870 non-null   datetime64[

In [7]:
conn = sqlite3.connect("crunchbase-investments.db")

chunk_iter=pd.read_csv("crunchbase-investments.csv", chunksize=5000,
                       encoding="iso-8859-1", usecols=useful_cols,
                      dtype=column_types, parse_dates=["funded_at"])

for chunk in chunk_iter:
    for col in category_cols.keys():
        if col in chunk.columns:
            chunk[col]=chunk[col].astype('category')
    chunk.to_sql('crunchbase_investments', conn, if_exists='append', index=True)

pd.read_sql('PRAGMA table_info(crunchbase_investments);', conn)

,cid,name,type,notnull,dflt_value,pk
0,0,index,INTEGER,0,None,0
1,1,company_name,TEXT,0,None,0
2,2,company_category_code,TEXT,0,None,0
3,3,company_country_code,TEXT,0,None,0
4,4,company_state_code,TEXT,0,None,0
5,5,company_region,TEXT,0,None,0
6,6,company_city,TEXT,0,None,0
7,7,investor_name,TEXT,0,None,0
8,8,investor_country_code,TEXT,0,None,0
9,9,investor_state_code,TEXT,0,None,0
